# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata_dict = json.loads(dataset.metadata.to_json())
print(f"{metadata_dict['name']}: {metadata_dict['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll get the list of all record sets, and for illustration, inspect one. All entities are referenced by their `@id`.

In [ ]:
# List available record sets by @id
record_sets = [rs['@id'] for rs in dataset.metadata.record_sets]
print("Available Record Sets (by @id):")
for rs in dataset.metadata.record_sets:
    print(f"@id: {rs['@id']}  |  name: {rs.get('name', '')}")

# Examine fields of the first record set as example
if len(dataset.metadata.record_sets) > 0:
    record_set0 = dataset.metadata.record_sets[0]
    fields = record_set0.get('field', [])
    print(f"\nFields in record set {record_set0['@id']}")
    for f in fields:
        # f can be an @id string, or a full field object (Croissant supports either)
        if isinstance(f, dict):
            print(f"Field @id: {f['@id']} | Name: {f.get('name', '')} | Data type: {f.get('dataType', '')}")
        else:
            print(f"Field @id: {f}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
dataframes = {}

# You can customize the list below if you want to load only specific record sets
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(dataframes[record_set_id])} rows for record set {record_set_id}")

# Print columns (fields) of the first record set's dataframe
first_rs = record_sets[0]
print("\nColumns in DataFrame for record set:", first_rs)
print(dataframes[first_rs].columns.tolist())
dataframes[first_rs].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping data. All field references are made by `@id`.

In [ ]:
# For demonstration, we choose a field from a known mental health assessment
record_set_id = record_sets[0]
df = dataframes[record_set_id].copy()

# Suppose the field '@id' for total GAD-7 score is 'GAD7_total_score' (replace with true @id from the overview above)
numeric_field = 'GAD7_total_score'  # Adjust this to actual @id as found in 'Fields in record set' above

if numeric_field in df.columns:
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}: (N={len(filtered_df)})")
    print(filtered_df.head())

    # Normalize the field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a demographic field, e.g., 'gender' (replace by real @id from fields list)
    group_field = 'gender'  # Adjust to actual @id as found in overview
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped mean {numeric_field} by {group_field}:")
        print(grouped_df.head())
else:
    print(f"Column '{numeric_field}' not found in data. Please insert the correct field @id from overview above.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example histogram of total GAD-7 score
if numeric_field in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
    plt.xlabel(numeric_field)
    plt.title(f'Distribution of {numeric_field}')
    plt.show()

    # Boxplot by group_field (e.g., gender)
    if group_field in df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.title(f'{numeric_field} by {group_field}')
        plt.show()
else:
    print(f"Column '{numeric_field}' not found in DataFrame. Please set the right field @id above.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset contains mental health survey responses from Kilifi County, Kenya, and includes standard assessment scores and demographic data.
- We loaded and visualized summary statistics and distributions for a numeric field (e.g., GAD-7 total score) with respect to demographic groups.
- For deeper insights, continue exploring additional record sets and fields using their `@id`s as demonstrated.
